In [ ]:
# CPU allocation controls. Set CPU_LIST like "0-15" before executing if desired.
import os

CPU_LIST = ""

def _parse_cpu_list(spec):
    cpus = []
    for part in str(spec).split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            lo, hi = part.split('-', 1)
            cpus.extend(range(int(lo), int(hi) + 1))
        else:
            cpus.append(int(part))
    return sorted(set(cpus))

if CPU_LIST and hasattr(os, 'sched_setaffinity'):
    os.sched_setaffinity(0, _parse_cpu_list(CPU_LIST))

print('effective CPU affinity:', sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else 'unavailable')


In [ ]:
from pathlib import Path
import json
import math

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'choi_covariance_cpu':
    REPO_ROOT = REPO_ROOT.parent

CAMPAIGN_ROOT = REPO_ROOT / 'choi_covariance_cpu' / 'cpu_data' / 'complex_particle_choi_transfer' / 'campaigns' / 'Nx16_Ny16-24-32_DW1_dwtrunc1_alpha1_sweep_S10_cycles2Ny_v5'
FIGURE_DIR = REPO_ROOT / 'choi_covariance_cpu' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ny_values = [16, 24, 32]
mode_ranks = [0, 1, 2]
alpha_values_for_mode_grids = [1.0, 2.0, 3.0]

mpl.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['CMU Sans Serif', 'DejaVu Sans'],
    'mathtext.fontset': 'cm',
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
})

def alpha_tag(alpha):
    return str(float(alpha)).replace('.', 'p').replace('-', 'm')


In [ ]:
manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'campaign manifest not found: {manifest_path}')
manifest = json.loads(manifest_path.read_text())
if manifest.get('status') != 'completed':
    raise RuntimeError(f"campaign is not complete yet: status={manifest.get('status')!r}")

records = []
for run_dir in sorted((CAMPAIGN_ROOT / 'runs').glob('N16x*_DW1_dwtrunc1_slab_a1-*')):
    summary_path = run_dir / 'run_summary.json'
    eig_path = run_dir / 'particle_choi_transfer_final_finite_eigenstates.npz'
    if not summary_path.exists() or not eig_path.exists():
        continue
    summary = json.loads(summary_path.read_text())
    records.append({
        'run_dir': run_dir,
        'eig_path': eig_path,
        'Nx': int(summary['Nx']),
        'Ny': int(summary['Ny']),
        'alpha_1': float(summary['alpha_1']),
        'status': summary.get('status'),
    })

print(f'loaded {len(records)} completed run records from {CAMPAIGN_ROOT}')
print(sorted((r['Ny'], r['alpha_1']) for r in records)[:5], '...')


In [ ]:
def build_low_mode_density_rows(alpha_1):
    rows = []
    for rec in records:
        if not math.isclose(rec['alpha_1'], float(alpha_1), rel_tol=0.0, abs_tol=1e-12):
            continue
        nx = rec['Nx']
        ny = rec['Ny']
        with np.load(rec['eig_path'], allow_pickle=False) as data:
            eigenstates = np.asarray(data['finite_eigenstates'], dtype=np.complex128)
            exponents = np.asarray(data['finite_eigenstate_exponents'], dtype=np.float64)
            valid_count = np.asarray(data['finite_eigenstate_valid_count_final'], dtype=np.int64)
            active_indices = np.asarray(data['active_top_layer_indices'], dtype=np.int64)
        xs = (active_indices // 2) % nx
        ys = active_indices // (2 * nx)
        dw_locs = [int(xs.min()), int(xs.max())]
        for rank in mode_ranks:
            density_maps = []
            abs_log_values = []
            for sample_idx in range(eigenstates.shape[0]):
                n_valid = int(valid_count[sample_idx])
                if n_valid <= rank:
                    continue
                sample_exponents = exponents[sample_idx, :n_valid]
                finite_mask = np.isfinite(sample_exponents)
                if finite_mask.sum() <= rank:
                    continue
                finite_order = np.where(finite_mask)[0]
                finite_order = finite_order[np.argsort(np.abs(sample_exponents[finite_order]))]
                eig_idx = int(finite_order[rank])
                vec = eigenstates[sample_idx, :, eig_idx]
                density_active = np.abs(vec) ** 2
                density_xy = np.zeros((nx, ny), dtype=float)
                np.add.at(density_xy, (xs, ys), density_active)
                norm = float(density_xy.sum())
                if np.isfinite(norm) and norm > 0:
                    density_maps.append(density_xy / norm)
                    abs_log_values.append(abs(float(sample_exponents[eig_idx])))
            if density_maps:
                stacked = np.asarray(density_maps, dtype=float)
                rows.append({
                    'Nx': nx,
                    'Ny': ny,
                    'alpha_1': float(alpha_1),
                    'rank': rank,
                    'density': stacked.mean(axis=0),
                    'n_samples': int(stacked.shape[0]),
                    'mean_abs_log_s': float(np.mean(abs_log_values)),
                    'median_abs_log_s': float(np.median(abs_log_values)),
                    'dw_locs': dw_locs,
                })
            else:
                rows.append({
                    'Nx': nx,
                    'Ny': ny,
                    'alpha_1': float(alpha_1),
                    'rank': rank,
                    'density': None,
                    'n_samples': 0,
                    'mean_abs_log_s': float('nan'),
                    'median_abs_log_s': float('nan'),
                    'dw_locs': dw_locs,
                })
    return rows

def plot_low_mode_density_grid(alpha_1):
    rows = build_low_mode_density_rows(alpha_1)
    finite_rows = [row for row in rows if row['density'] is not None]
    vmax = max((float(row['density'].max()) for row in finite_rows), default=1.0)
    fig, axes = plt.subplots(len(mode_ranks), len(ny_values), figsize=(6.75, 5.9), sharex=False, sharey=False, constrained_layout=True)
    last_im = None
    for col, ny in enumerate(ny_values):
        for row_idx, rank in enumerate(mode_ranks):
            ax = axes[row_idx, col]
            matches = [item for item in rows if item['Ny'] == ny and item['rank'] == rank]
            item = matches[0] if matches else None
            if item is None or item['density'] is None:
                ax.text(0.5, 0.5, 'no finite mode', ha='center', va='center', transform=ax.transAxes, fontsize=7)
                ax.set_axis_off()
                continue
            density = item['density']
            last_im = ax.imshow(
                density.T,
                origin='lower',
                aspect='auto',
                interpolation='nearest',
                extent=(-0.5, item['Nx'] - 0.5, -0.5, item['Ny'] - 0.5),
                cmap='magma',
                vmin=0.0,
                vmax=vmax,
            )
            for dw in item['dw_locs']:
                ax.axvline(dw, color='cyan', linestyle='--', linewidth=0.8, alpha=0.85)
            if row_idx == 0:
                ax.set_title(fr'$N_y={ny}$')
            if col == 0:
                ax.set_ylabel(fr'rank {rank + 1}\n$|\log s|$={item["mean_abs_log_s"]:.3g}\n$y$')
            if row_idx == len(mode_ranks) - 1:
                ax.set_xlabel(r'$x$')
            ax.set_xlim(-0.5, item['Nx'] - 0.5)
            ax.set_ylim(-0.5, item['Ny'] - 0.5)
            ax.tick_params(labelsize=6)
    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.82, pad=0.015)
        cbar.set_label(r'sample-mixed normalized $\sum_\mu |\psi_\mu(x,y)|^2$')
    fig.suptitle(fr'$N_x=16$, $\alpha_1={alpha_1:g}$: final low-$|\log s|$ Choi modes', y=1.015, fontsize=9)
    png_path = FIGURE_DIR / f'nx16_alpha1_sweep_low_log_mode_density_alpha{alpha_tag(alpha_1)}.png'
    pdf_path = FIGURE_DIR / f'nx16_alpha1_sweep_low_log_mode_density_alpha{alpha_tag(alpha_1)}.pdf'
    fig.savefig(png_path, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    print(f'saved {png_path}')
    print(f'saved {pdf_path}')
    for item in rows:
        if item['density'] is None:
            print(f"alpha={alpha_1:g}, Ny={item['Ny']}, rank={item['rank'] + 1}: no finite mode")
        else:
            peak = np.unravel_index(int(np.argmax(item['density'])), item['density'].shape)
            print(
                f"alpha={alpha_1:g}, Ny={item['Ny']}, rank={item['rank'] + 1}: "
                f"n={item['n_samples']}, mean |log s|={item['mean_abs_log_s']:.4g}, peak=(x={peak[0]}, y={peak[1]}), "
                f"DW={item['dw_locs']}"
            )
    return fig, rows


In [ ]:
fig_alpha1, rows_alpha1 = plot_low_mode_density_grid(1.0)
plt.show()


In [ ]:
fig_alpha2, rows_alpha2 = plot_low_mode_density_grid(2.0)
plt.show()


In [ ]:
fig_alpha3, rows_alpha3 = plot_low_mode_density_grid(3.0)
plt.show()
